In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.msl_al_target_list
-- group by 1 order by 2 desc

In [0]:
with msl_target_with_affiliations as (
  select distinct a.HCP_NPI, b.hco_npi
  from com_edp_prd.cmpa_insights_internal_schema.msl_al_target_list as a
  left join com_edp_prd.cmpa_insights_internal_schema.reference_file as b
    on cast(a.HCP_NPI as string) = b.hcp_npi
)
select *
from msl_target_with_affiliations

In [0]:
select *
from com_edp_prd.cmpa_insights_internal_schema.reference_file
where hcp_npi in ('1770949901', '1942545314', '1891794525')

In [0]:

-- HCO-LEVEL PATIENT COUNT AGGREGATION (STANDALONE)
-- Logic: MAX(sum of most recent HCP surveys, most recent HCO survey)


WITH survey_responses_with_dates AS (
  -- Get all survey responses with dates from source tables
  SELECT DISTINCT
    a.modified_date__v,
    a.order__v AS question_order,
    a.question_text__v AS question_text,
    COALESCE(
      CAST(a.response__v AS STRING), 
      CAST(a.text__v AS STRING), 
      CAST(a.number__v AS STRING)
    ) AS question_response,
    b.name__v AS survey_name,
    c.account_display_name__v AS respondent_name
  FROM com_edp_prd.com_intgr.question_response a
  LEFT JOIN com_edp_prd.com_intgr.survey b
    ON a.ctrl_survey__v = b.id
  LEFT JOIN com_edp_prd.com_intgr.survey_target c
    ON a.survey_target__v = c.id
  WHERE b.name__v IN ('Denali HCP Survey', 'Denali HCO Survey')
    AND (a.response__v IS NOT NULL 
         OR a.text__v IS NOT NULL 
         OR a.number__v IS NOT NULL)
    AND (
      a.question_text__v ILIKE '%How many Hunter Syndrome patients are they currently treating?%'
      OR a.question_text__v ILIKE '%How many Hunter Syndrome patients does this site support?%'
    )
),

hcp_hco_mapping AS (
  -- Get HCP-HCO relationships from vw_HCP_HCO_Territory
  SELECT DISTINCT
    HCP,
    HCO,
    Territory
  FROM vw_HCP_HCO_Territory
  WHERE HCP IS NOT NULL 
    AND HCO IS NOT NULL
),

survey_base AS (
  -- Join survey responses with HCP-HCO relationships
  SELECT 
    s.question_response,
    s.respondent_name,
    s.survey_name,
    s.modified_date__v,
    CAST(s.question_response AS INT) AS patient_count,
    -- For HCP surveys, get HCO from mapping
    CASE 
      WHEN s.survey_name = 'Denali HCP Survey' THEN m.HCO
      WHEN s.survey_name = 'Denali HCO Survey' THEN s.respondent_name
      ELSE NULL
    END AS HCO,
    -- For HCP surveys, keep HCP name
    CASE 
      WHEN s.survey_name = 'Denali HCP Survey' THEN s.respondent_name
      ELSE NULL
    END AS HCP
  FROM survey_responses_with_dates s
  LEFT JOIN hcp_hco_mapping m
    ON s.respondent_name = m.HCP
  WHERE s.question_response IS NOT NULL
),

-- STEP 1: GET MOST RECENT HCP SURVEY RESPONSE PER HCP
most_recent_hcp_surveys AS (
  SELECT 
    HCP,
    HCO,
    patient_count,
    modified_date__v,
    ROW_NUMBER() OVER (PARTITION BY HCP, HCO ORDER BY modified_date__v DESC) AS rn
  FROM survey_base
  WHERE survey_name = 'Denali HCP Survey'
    AND HCP IS NOT NULL
    AND HCO IS NOT NULL
),

-- STEP 2: SUM MOST RECENT HCP SURVEYS BY HCO
hcp_sum_by_hco AS (
  SELECT 
    HCO,
    SUM(patient_count) AS hcp_patient_sum,
    COUNT(DISTINCT HCP) AS num_hcps_responded
  FROM most_recent_hcp_surveys
  WHERE rn = 1  -- Only most recent response per HCP
  GROUP BY HCO
),

-- STEP 3: GET MOST RECENT HCO SURVEY RESPONSE PER HCO
most_recent_hco_surveys AS (
  SELECT 
    HCO,
    patient_count AS hco_patient_count,
    modified_date__v AS hco_survey_date,
    ROW_NUMBER() OVER (PARTITION BY HCO ORDER BY modified_date__v DESC) AS rn
  FROM survey_base
  WHERE survey_name = 'Denali HCO Survey'
    AND HCO IS NOT NULL
),

-- STEP 4: COMBINE AND TAKE MAX
combined_counts AS (
  SELECT 
    COALESCE(hcp.HCO, hco.HCO) AS HCO,
    COALESCE(hcp.hcp_patient_sum, 0) AS hcp_sum,
    COALESCE(hcp.num_hcps_responded, 0) AS num_hcps_responded,
    COALESCE(hco.hco_patient_count, 0) AS hco_recent,
    hco.hco_survey_date,
    GREATEST(
      COALESCE(hcp.hcp_patient_sum, 0),
      COALESCE(hco.hco_patient_count, 0)
    ) AS final_patient_count
  FROM hcp_sum_by_hco hcp
  FULL OUTER JOIN (
    SELECT HCO, hco_patient_count, hco_survey_date
    FROM most_recent_hco_surveys
    WHERE rn = 1  -- Only most recent HCO survey
  ) hco ON hcp.HCO = hco.HCO
)

-- FINAL OUTPUT: HCO-LEVEL PATIENT COUNTS
SELECT 
  HCO,
  hcp_sum AS sum_of_most_recent_hcp_surveys,
  num_hcps_responded AS number_of_hcps_with_surveys,
  hco_recent AS most_recent_hco_survey,
  hco_survey_date AS hco_survey_date,
  final_patient_count,
  CASE 
    WHEN hcp_sum > hco_recent THEN 'HCP_SUM'
    WHEN hco_recent > hcp_sum THEN 'HCO_SURVEY'
    WHEN hcp_sum = hco_recent AND hcp_sum > 0 THEN 'EQUAL'
    ELSE 'NO_DATA'
  END AS source_used
FROM combined_counts
WHERE HCO IS NOT NULL
ORDER BY final_patient_count DESC, HCO ASC;

In [0]:
select * from com_edp_prd.com_consm.vw_emp_roster_crx_outbound